In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from torch.utils.data import DataLoader

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint

import matplotlib.pyplot as plt

from src.reader import KineticDataset, KineticDatasetVideo, KittiVideoDataset

from src.my_model import MaskVideo, Encoder, Decoder
from src.trainer import AttentionMaskModeling

import torch

from torch.profiler import profile, ProfilerActivity, record_function

In [3]:
path = "/mnt/e/Kitti/"
split = "training"

ds = KittiVideoDataset.get_ds(path, split, 16, size=(256, 256), n_samples=100)
ds_loader = DataLoader(
    ds, batch_size=1, shuffle=True,
    num_workers=4, pin_memory=True
)

In [ ]:
for image in ds:
    print(image.shape)
    break


In [4]:
is_finetune = False
image_size = (256, 256)
patch_size = (8, 8)
depth, heads, dim, embed_dim = 8, 8, 512, 32

window_size = (3, 3)
length, height, width = 32, 32, 32
temporal_depth, temporal_heads, temporal_dim = 4, 8, 128
n_codes = 8192

vitvq_path = "./checkpoint/imagenet_vitvq_small.ckpt"

In [5]:
model = MaskVideo(
    is_finetune=is_finetune,
    
    image_size=image_size, patch_size=patch_size,
    vit_depth=depth, vit_heads=heads, dim=dim,
    n_codes=n_codes, embed_dim=embed_dim,

    window_size=window_size,
    length=length, height=height, width=width,
    temporal_depth=temporal_depth, temporal_heads=temporal_heads,

    drop_prob=0.1, depth_prob=0.1,
    vitvq_path=vitvq_path
)
model.cuda()
print()

In [6]:
for batch in ds_loader:
    print(batch.shape)

    # output = model(batch.cuda())

    break

torch.Size([1, 16, 3, 256, 256])


In [8]:
encoder = Encoder(
    image_size=image_size, patch_size=patch_size,
    depth=temporal_depth, heads=heads, dim=dim,
    length=length,
    vit=model.vit,
    pre_quant=model.pre_quant,
    quantizer=model.quantizer,
    cls_token=model.cls_token,
    mask_token=model.mask_token,
    transformer=model.transformer,
    transformer_norm=model.transformer_norm
)
encoder.cuda()
print()

In [23]:
# ks, vs = torch.randn(4, 1024, 8, 15, 64).cuda(), torch.randn(4, 1024, 8, 15, 64).cuda()
ks, vs = None, None
ks, vs = torch.empty((4, 1024, 8, 15, 64)).cuda(), torch.empty((4, 1024, 8, 15, 64)).cuda()
cur_t = 0
with profile(activities=[ProfilerActivity.CUDA], record_shapes=True) as prof:
    for t in range(batch.shape[1]):
        frame = batch[:, t:t+1]

        with record_function("model_inference"):
            code, k, v = encoder(
            # code = encoder(
                frame.cuda(),
                ks[:, :, :, :t], vs[:, :, :, :t]
            )
            # ks = torch.cat([ks, k], dim=3) if ks is not None else k
            # vs = torch.cat([vs, v], dim=3) if vs is not None else v
            # print(ks.shape, k.shape)
            ks[:, :, :, cur_t].copy_(k[:, :, :, 0])
            vs[:, :, :, cur_t].copy_(v[:, :, :, 0])
            cur_t += 1

        if t == 4: break
        else: print("-"*30)

temporal_mask torch.Size([2, 2])
------------------------------
temporal_mask torch.Size([2, 3])
------------------------------
temporal_mask torch.Size([2, 4])
------------------------------
temporal_mask torch.Size([2, 5])
------------------------------
temporal_mask torch.Size([2, 6])


In [ ]:
torch.onnx.export(
    encoder,
    {
        "frames": torch.randn(1, 1, 3, 256, 256).cuda(),
        "ks": torch.randn(4, 1024, 8, 15, 64).cuda(),
        "vs": torch.randn(4, 1024, 8, 15, 64).cuda(),
    },
    "encoder_small_v2.onnx",
    export_params=True
)
print("ONNX model saved: encoder_small_v2.onnx")
# trtexec --onnx=encoder_small.onnx --saveEngine=encoder_small.trt --fp16

/tmp/ipykernel_174241/143516844.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


temporal_mask torch.Size([2, 17])
ONNX model saved: encoder_small_v2.onnx


In [ ]:
import torch.nn.functional as F

In [ ]:
n_heads = 8
head_dim = 512/n_heads
window_size = (3, 3)
height, width = 256, 256

_padding_size = (window_size[0]//2, window_size[1]//2)
_size = window_size[0] * window_size[1]

# (BT, n_heads, HW, D')
def get_neighbor(x):
    """
        1/ flatten batch, time, and head dimensions
        2/ 
    """
    # print("x", x.shape)
    BT, _, HW, _ = x.shape

    _x = x.reshape(BT, n_heads, height, width, head_dim)
    _x = _x.flatten(0, 1)  # (BT*n_heads, H, W, D')

    neighbor_x = F.unfold(
        _x.permute(0, 3, 1, 2), window_size,
        padding=_padding_size, stride=1,
    )  # (BT*n_heads, D*9, HW)
    neighbor_x = neighbor_x.reshape(BT, head_dim, n_heads, _size, HW)
    neighbor_x = neighbor_x.permute(0, 4, 2, 3, 1).flatten(0, 1)
    
    return neighbor_x

B, T, HW = 1, 4, 16

q = torch.rand((B*T, n_heads, HW, head_dim))
neighbor_q = get_neighbor(q)
